# 9장 (실습 1): Windows Server와 Ubuntu 서버의 OS·SW 버전 자동 수집하기

> 책 『AI 보안, 클라우드에서 시작한다』 9장 [실습] 1번을 노트북에서 단계별로 실행하는 버전입니다. 9-1에서 패치 자동화의 전제는 **현황관리(SW 버전정보 수집)** 라고 했습니다. 서버에서 OS와 설치된 SW의 버전을 사람 손을 거치지 않고 수집하고, 여러 서버의 결과를 하나로 합쳐 기준 버전과 대조합니다. 소요 시간은 40~60분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. OS 패키지 관리자(dpkg)에 **보이지 않는** WAS·pip 라이브러리·단일 실행 파일을 파일 시스템 SBOM 스캔과 실행 파일 검사로 찾아냅니다.
2. 호스트명이 같은 서버가 있어도 **기기 고유 식별자(machine-id)** 로 구분하는 이유를 확인합니다.
3. 여러 서버의 결과를 합쳐 **기준 버전 목록**과 자동으로 대조하고, 이름을 정확히 맞추는 일의 중요성을 확인합니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 구성 | 예정 |
| 1 | (선택) Windows에서 수집 | 예정 |
| 2 | 실습용 Ubuntu 서버 만들기 | 예정 |
| 3 | Ubuntu 서버에서 수집 | 예정 |
| 4 | 여러 서버의 결과를 합치고 기준 버전과 대조 | 예정 |
| 5 | 매일 자동으로 돌게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

## 실습 시나리오

README의 실습은 Windows PC와 WSL의 Ubuntu에 직접 Tomcat 등을 설치하고 `sudo`로 수집합니다. 노트북에서는 `sudo` 암호를 입력할 수 없고, 실습용 설치가 내 WSL에 남지 않게 하려고 **"실습용 Ubuntu 서버"를 Docker 컨테이너로 만들어** 그 안에서 root로 설치·수집합니다. 그리고 그 서버를 복제한 **같은 호스트명의 서버**를 하나 더 만들어, 복제 VM에서 흔히 생기는 호스트명 중복 문제를 재현합니다.

```text
 web01 (컨테이너, Ubuntu 24.04)        web01-clone (web01을 복제, 호스트명도 web01)
   ├─ /opt/tomcat (압축 설치 WAS)         └─ urllib3만 수정 버전으로 올림
   ├─ /opt/app-venv (pip 라이브러리)
   └─ /usr/local/bin/gitleaks (단일 실행 파일)
         │ collect_linux.sh                    │ collect_linux.sh
         ▼                                     ▼
   inventories/inventory-web01-<id>.json   inventories/inventory-web01-<id>.json
         └──────────── merge_report.py + watchlist.csv ────────────▶ 통합 목록, 기준 버전 미달
```

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Docker와 Trivy(8장 [실습 준비]), 인터넷 연결이 필요합니다. 1단계(선택)는 WSL에서 Windows의 PowerShell을 불러 실행합니다.
>
> **2. Jupyter 준비하기 (WSL 터미널에서 한 번만)**<br>
> - 2장 노트북([ch02_lab1_container_isolation.ipynb](../ch02-container-security/ch02_lab1_container_isolation.ipynb))의 실습 전 확인사항과 같습니다. `~/.venv-lab`에 jupyterlab을 설치한 뒤 이 폴더에서 `~/.venv-lab/bin/jupyter lab`을 실행합니다.
> - 노트북은 **반드시 `ch09-ops-tracing-sw-inventory` 폴더에서 열어야** 합니다. 실습 파일을 이 폴더 기준 경로로 읽습니다.
>
> **3. 기준 버전**<br>
> - 이 노트북은 2026년 10월 23일 Ubuntu 24.04(WSL2), Docker 29.1.3, Trivy 0.76.0, 컨테이너 ubuntu:24.04, Tomcat 9.0.80, Gitleaks 8.24.0에서 끝까지 실행해 확인했습니다. 실행 결과가 저장된 버전은 `ch09_lab1_sw_inventory_executed.ipynb`입니다.

## 0. 실습 준비와 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 구성 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | (선택) Windows에서 수집 | 예정 |
| 2 | 실습용 Ubuntu 서버 만들기 | 예정 |
| 3 | Ubuntu 서버에서 수집 | 예정 |
| 4 | 여러 서버의 결과를 합치고 기준 버전과 대조 | 예정 |
| 5 | 매일 자동으로 돌게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "docker": "docker --version",
    "trivy": "trivy --version | head -1",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

| 파일 | 하는 일 |
|---|---|
| `sw-inventory/collect_windows.ps1` | Windows의 OS 버전, 보안 업데이트(핫픽스), 레지스트리에 등록된 설치 SW, 포트를 열고 대기 중인 프로세스의 실행 파일 버전을 수집 |
| `sw-inventory/collect_linux.sh` | Ubuntu의 OS와 커널, dpkg·snap 패키지, 업데이트 대기 목록, 웹 서버·WAS·pip 등 패키지 관리자 밖 SW(파일 시스템 SBOM 스캔), 그 밖의 실행 파일(해시), 실행 중인 컨테이너 이미지를 수집 |
| `sw-inventory/list_nonpkg.py` | 수집 결과에서 OS 패키지 관리자 밖에 설치된 SW만 골라 보여 줌 |
| `sw-inventory/merge_report.py` | 여러 서버의 수집 결과(JSON)를 하나의 CSV로 합치고, 기준 버전 목록과 대조 |
| `sw-inventory/watchlist.csv` | 기준 버전 목록 (SW 이름 정규식, 최소 버전, 사유) |

두 수집 스크립트는 9-1의 영역 구분을 그대로 따릅니다. **OS 패키지**(레지스트리, dpkg)는 패키지 관리자의 기록으로, **운영자가 직접 설치한 SW**는 대기 중인 프로세스와 /opt·/usr/local 경로로 찾습니다. 그리고 호스트명은 겹칠 수 있으므로, 모든 결과에 기기 고유 식별자(Windows의 MachineGuid, Linux의 machine-id)를 함께 남깁니다.

실습 파일을 `~/sw-inventory-lab`으로 복사합니다. 컨테이너들이 이 폴더를 `/lab`으로 함께 보고, 수집 결과를 `inventories/`에 남깁니다.

In [ ]:
WORK = Path.home() / "sw-inventory-lab"
sh(f"rm -rf {WORK} && cp -r '{LAB / 'sw-inventory'}' {WORK} && chmod +x {WORK}/collect_linux.sh")
print(sorted(p.name for p in WORK.iterdir()))
print((WORK / "watchlist.csv").read_text())

## 1. (선택) Windows에서 수집

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 구성 | ✅ 완료 |
| 1 | (선택) Windows에서 수집 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 실습용 Ubuntu 서버 만들기 | 예정 |
| 3 | Ubuntu 서버에서 수집 | 예정 |
| 4 | 여러 서버의 결과를 합치고 기준 버전과 대조 | 예정 |
| 5 | 매일 자동으로 돌게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

Windows PC(또는 Windows Server)에서는 PowerShell 수집 스크립트를 씁니다. WSL에서는 `powershell.exe`로 Windows의 PowerShell을 바로 부를 수 있습니다.

> ⚠️ 이 셀은 **기본값으로 실행하지 않습니다**. 실행하면 내 Windows PC의 설치 프로그램 목록과 포트를 연 프로세스 목록이 노트북 출력과 결과 파일에 남습니다. 노트북을 다른 사람과 공유할 계획이라면 출력을 지운 뒤 공유하세요. 실행하려면 `RUN_WINDOWS = True`로 바꿉니다. 관리자 권한이 아니면 포트를 연 프로세스의 경로 일부가 빠질 수 있습니다.

In [ ]:
RUN_WINDOWS = False

if RUN_WINDOWS:
    ps1 = sh(f"wslpath -w {WORK / 'collect_windows.ps1'}")
    out = sh(f"wslpath -w {WORK / 'inventories'}")
    # Windows PowerShell의 출력 인코딩을 UTF-8로 맞춰야 한글이 깨지지 않는다
    cmd = ["powershell.exe", "-NoProfile", "-ExecutionPolicy", "Bypass", "-Command",
           f"[Console]::OutputEncoding=[Text.Encoding]::UTF8; & '{ps1}' -OutDir '{out}'"]
    print(subprocess.run(cmd, capture_output=True, encoding="utf-8", errors="replace").stdout)
    for f in (WORK / "inventories").glob("inventory-*.json"):
        inv = json.loads(f.read_text(encoding="utf-8-sig"))
        if "Windows" in inv["os"]["name"]:
            from collections import Counter
            table(["수집 경로(source)", "개수"], Counter(p["source"] for p in inv["packages"]).most_common())
            table(["포트 대기 프로세스", "버전"], [(p["name"], p.get("version", "")) for p in inv["packages"]
                                               if p["source"] == "listening-process"])
else:
    print("RUN_WINDOWS = False: 건너뜀")

Windows에서는 OS 버전, 핫픽스, 레지스트리에 등록된 설치 SW, 포트를 열고 대기 중인 프로세스와 그 실행 파일 버전이 수집됩니다. 레지스트리 목록만 보면 수십 개 SW 사이에 묻히지만, "포트를 열고 있는 SW"로 따로 뽑으면 먼저 확인해야 할 대상이 바로 드러납니다(9-3의 워터링홀 공격).

> **수집 스크립트가 보는 곳**
> - `Win32_OperatingSystem`: OS 이름, 버전, 빌드 번호, 마지막 부팅 시각
> - `Get-HotFix`: 설치된 보안 업데이트와 설치 날짜(최근 20개)
> - 레지스트리 `HKLM\...\Uninstall` (64비트·32비트): 설치 프로그램 이름, 버전, 게시자
> - `Get-NetTCPConnection -State Listen`: 포트를 열고 대기 중인 프로세스와 그 실행 파일의 제품 버전

## 2. 실습용 Ubuntu 서버 만들기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 구성 | ✅ 완료 |
| 1 | (선택) Windows에서 수집 | ✅ 완료 |
| 2 | 실습용 Ubuntu 서버 만들기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | Ubuntu 서버에서 수집 | 예정 |
| 4 | 여러 서버의 결과를 합치고 기준 버전과 대조 | 예정 |
| 5 | 매일 자동으로 돌게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

### 2.1. 서버 띄우기

Ubuntu 24.04 컨테이너를 `web01`이라는 호스트명으로 띄웁니다. 수집 스크립트가 쓰는 Trivy는 내 WSL의 것을 읽기 전용으로 빌려 주고, Trivy의 취약점 DB 캐시는 실습 전용 볼륨에 둡니다.

> 📌 (참고) 컨테이너에는 `/etc/machine-id`가 비어 있을 수 있습니다. 실제 서버처럼 무작위 ID를 하나 만들어 넣습니다.

In [ ]:
%%bash
docker rm -f web01 web01-clone >/dev/null 2>&1
docker pull -q ubuntu:24.04 >/dev/null
docker run -d --name web01 --hostname web01 -v ~/sw-inventory-lab:/lab \
  -v "$(command -v trivy)":/usr/bin/trivy:ro -v sw-lab-trivy-cache:/root/.cache/trivy \
  ubuntu:24.04 sleep infinity >/dev/null
docker exec web01 bash -c '
  apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq python3 python3-venv wget ca-certificates >/dev/null 2>&1
  cat /proc/sys/kernel/random/uuid | tr -d - > /etc/machine-id
  . /etc/os-release; echo "$(hostname) | $PRETTY_NAME | machine-id $(cut -c1-8 /etc/machine-id)"'

### 2.2. 패키지 관리자를 거치지 않는 설치 세 가지

실제 서버처럼 만들기 위해, OS 패키지 관리자를 거치지 않는 설치 세 가지를 만들어 둡니다. 압축 파일로 설치한 WAS(Tomcat), pip로 설치한 Python 라이브러리, 내려받아 복사한 단일 실행 파일(gitleaks)입니다. 모두 일부러 오래된 버전입니다.

In [ ]:
%%bash
docker exec web01 bash -c '
  # WAS를 압축 파일로 설치 (일부러 오래된 버전)
  cd /opt && wget -q https://archive.apache.org/dist/tomcat/tomcat-9/v9.0.80/bin/apache-tomcat-9.0.80.tar.gz
  tar xzf apache-tomcat-9.0.80.tar.gz && mv apache-tomcat-9.0.80 tomcat && rm apache-tomcat-9.0.80.tar.gz
  # pip 가상환경에 오래된 라이브러리 설치
  python3 -m venv /opt/app-venv
  /opt/app-venv/bin/pip install -q requests==2.25.1 urllib3==1.26.4 jinja2==3.1.2 werkzeug==2.2.2 2>&1 | grep -iv notice
  # 단일 실행 파일을 내려받아 복사 (공식 체크섬과 대조한 뒤 설치)
  cd /tmp && V=8.24.0
  wget -q https://github.com/gitleaks/gitleaks/releases/download/v${V}/gitleaks_${V}_linux_x64.tar.gz
  wget -q https://github.com/gitleaks/gitleaks/releases/download/v${V}/gitleaks_${V}_checksums.txt
  grep linux_x64.tar.gz gitleaks_${V}_checksums.txt | sha256sum -c
  tar xzf gitleaks_${V}_linux_x64.tar.gz gitleaks && mv gitleaks /usr/local/bin/
  ls -d /opt/tomcat /opt/app-venv /usr/local/bin/gitleaks'

### 2.3. 복제 서버 만들기

web01을 그대로 복제해 `web01-clone`을 만듭니다. VM을 복제하면 호스트명까지 그대로 따라오는 경우가 많으므로 **호스트명도 `web01`** 로 둡니다. machine-id만 새로 만들고, 이 서버에서는 누군가 urllib3를 수정 버전으로 올려 두었다고 합시다.

In [ ]:
%%bash
docker commit web01 sw-lab-server:clone >/dev/null
docker run -d --name web01-clone --hostname web01 -v ~/sw-inventory-lab:/lab \
  -v "$(command -v trivy)":/usr/bin/trivy:ro -v sw-lab-trivy-cache:/root/.cache/trivy \
  sw-lab-server:clone sleep infinity >/dev/null
docker exec web01-clone bash -c '
  cat /proc/sys/kernel/random/uuid | tr -d - > /etc/machine-id
  /opt/app-venv/bin/pip install -q urllib3==1.26.19 2>&1 | grep -iv notice
  echo "$(hostname) | machine-id $(cut -c1-8 /etc/machine-id) | urllib3 $(/opt/app-venv/bin/pip show urllib3 | grep Version)"'

## 3. Ubuntu 서버에서 수집

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 구성 | ✅ 완료 |
| 1 | (선택) Windows에서 수집 | ✅ 완료 |
| 2 | 실습용 Ubuntu 서버 만들기 | ✅ 완료 |
| 3 | Ubuntu 서버에서 수집 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 여러 서버의 결과를 합치고 기준 버전과 대조 | 예정 |
| 5 | 매일 자동으로 돌게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

### 3.1. dpkg는 Tomcat을 모른다

OS 패키지 관리자에게 Tomcat의 핵심 파일이 어느 패키지 소속인지 물어봅니다.

In [ ]:
%%bash
docker exec web01 dpkg -S /opt/tomcat/lib/catalina.jar || true

`dpkg -S`가 "no path found"라고 답합니다. OS 패키지 기준으로만 보면 Tomcat은 이 서버에 "없는 것"입니다.

### 3.2. 수집 스크립트 실행

두 서버에서 수집 스크립트를 root로 실행합니다. 결과는 `/lab/inventories`(= `~/sw-inventory-lab/inventories`)에 남고, 수집이 끝나면 결과 파일의 소유자를 내 사용자로 바꿔 둡니다. 처음에는 Trivy가 취약점 DB를 내려받느라 1~2분 걸립니다.

In [ ]:
%%bash
for s in web01 web01-clone; do
  echo "── $s"
  docker exec $s bash -c "cd /lab && ./collect_linux.sh /lab/inventories && chown -R $(id -u):$(id -g) /lab/inventories"
done

### 3.3. 패키지 관리자 밖의 SW만 골라 보기

In [ ]:
import sys

mid = sh("docker exec web01 cut -c1-8 /etc/machine-id")
print(f"web01의 machine-id 앞자리: {mid}\n")
print(subprocess.run([sys.executable, "list_nonpkg.py", mid], cwd=WORK, capture_output=True, text=True).stdout)

dpkg는 Tomcat을 찾지 못하지만, 수집 스크립트는 SBOM 스캔으로 Tomcat 9.0.80과 pip 패키지의 버전을, 실행 파일 검사로 gitleaks를 찾아냅니다. 결과의 출처별 개수를 셉니다.

In [ ]:
from collections import Counter


def inventory(machine_prefix):
    f = next((WORK / "inventories").glob(f"*{machine_prefix}*.json"))
    return json.loads(f.read_text(encoding="utf-8-sig"))


def examples(source):
    return ", ".join(sorted({p["name"].split(":")[-1] for p in inv["packages"] if p["source"] == source})[:4])


inv = inventory(mid)
table(["출처(source)", "개수", "예"], [(s, n, examples(s)) for s, n in Counter(p["source"] for p in inv["packages"]).most_common()])

`[표 실습9-1]` 수집 결과의 출처별 구분

| 출처(source) | 어떻게 찾았나 |
|---|---|
| `dpkg`, `snap` | OS 패키지 관리자의 기록 |
| `sbom-jar`, `sbom-python-pkg` | /opt, /srv, /usr/local, /var/www를 Trivy로 파일 시스템 스캔. 실행하지 않고 JAR의 메타데이터와 Python의 dist-info를 읽음 |
| `manual` | 패키지 관리자에도 SBOM 스캔에도 잡히지 않은 실행 파일 (버전 확인 필요, 해시 기록) |

SBOM으로 버전이 확인된 폴더(/opt/tomcat, /opt/app-venv) 안의 시작 스크립트 같은 실행 파일은 그 SW의 일부이므로 따로 세지 않습니다.

> 📌 (참고) 위 표에 `sbom-gobinary`가 보인다면, Trivy가 Go로 만든 실행 파일(gitleaks) 안에 기록된 **빌드 의존성 목록**까지 읽은 것입니다. gitleaks 자체의 버전은 알 수 없어 `manual`로 남지만, 그 안에 든 라이브러리의 취약점은 이 목록으로 찾을 수 있습니다. Trivy 버전에 따라 이 항목이 나오지 않을 수도 있습니다.

> **왜 실행해서 버전을 확인하지 않을까?**
> 실행 파일에 `--version`을 붙여 실행하면 버전을 쉽게 얻을 수 있습니다. 그러나 수집 스크립트는 보통 root 권한으로 돌기 때문에, 출처를 모르는 파일을 root로 실행하는 것은 그 자체가 공격 경로가 됩니다. 그래서 수집 단계에서는 실행하지 않고, 메타데이터를 읽거나(SBOM 스캔) 경로·해시·수정 시각만 남깁니다. 같은 해시가 여러 서버에서 나오면 한 번만 확인하면 됩니다.

In [ ]:
table(["이름", "경로", "SHA-256(앞 16자)", "수정일"],
      [(p["name"], p["path"], p["sha256"], p["modified"]) for p in inv["packages"] if p["source"] == "manual"])

## 4. 여러 서버의 결과를 합치고 기준 버전과 대조

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 구성 | ✅ 완료 |
| 1 | (선택) Windows에서 수집 | ✅ 완료 |
| 2 | 실습용 Ubuntu 서버 만들기 | ✅ 완료 |
| 3 | Ubuntu 서버에서 수집 | ✅ 완료 |
| 4 | 여러 서버의 결과를 합치고 기준 버전과 대조 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 매일 자동으로 돌게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

두 서버의 결과를 합치고 기준 버전 목록과 대조합니다. 1단계에서 Windows 결과를 만들었다면 함께 합쳐집니다.

In [ ]:
print(subprocess.run([sys.executable, "merge_report.py", "inventories", "watchlist.csv"], cwd=WORK,
                     capture_output=True, text=True).stdout)

결과에서 세 가지를 확인합니다.

- **호스트명 중복**: 두 서버 모두 호스트명이 `web01`입니다. 호스트명만으로 자산을 구분했다면 두 서버의 결과가 서로 덮어써졌을 것입니다. 9-1의 "유일한 식별자" 조건이 왜 필요한지 보여 주는 장면입니다. 실제 회사에서도 온프레미스와 클라우드, 복제한 VM 사이에서 같은 호스트명이 자주 생깁니다. (README에서는 WSL의 Ubuntu와 Windows가 같은 호스트명을 쓰는 것으로 같은 문제를 보였습니다.)
- **기준 미달 목록에도 호스트명만 있습니다**: 위 출력의 "기준 버전 미달" 줄에는 `web01`만 찍혀 있어, urllib3가 미달인 서버가 두 대 중 어느 것인지 알 수 없습니다. 보고서의 모든 줄에 machine-id를 함께 남겨야 합니다. 아래 4.1에서 통합 목록으로 확인합니다.
- **OS 패키지 밖 SW의 기준 버전 미달**: 압축 설치한 Tomcat 9.0.80은 CISA KEV에 등재된 CVE-2025-24813의 수정 버전(9.0.99)보다 낮고, pip로 설치한 urllib3도 web01에서는 수정 버전보다 낮습니다. dpkg만 봤다면 둘 다 보이지 않았을 취약점입니다.
- **OS 패키지의 기준 버전 미달**: `watchlist.csv`의 curl·libc6 기준은 Ubuntu 보안 저장소의 수정 버전입니다. 컨테이너 이미지는 최근에 만들어진 것이라 이번 결과에는 걸리지 않을 수 있습니다.

### 4.1. 통합 목록에서 "어느 서버에 있나" 찾기

통합 CSV는 "이 SW가 어느 서버에 있나"에 바로 답하는 표입니다. Tomcat과 urllib3를 찾아봅니다. 호스트명은 같아도 machine-id로 보면 urllib3를 올려 둔 복제 서버와 그대로인 원래 서버가 구분됩니다.

In [ ]:
import csv

rows = list(csv.DictReader(open(WORK / "inventory-all.csv", encoding="utf-8-sig")))
print(f"통합 목록: {len(rows)}행, 서버 {len({r['machine_id'] for r in rows})}대 (호스트명 {len({r['host'] for r in rows})}개)")
table(["호스트명", "machine-id", "SW", "버전", "출처"],
      [(r["host"], r["machine_id"][:8], r["software"], r["version"], r["source"]) for r in rows
       if r["software"].split(":")[-1] in ("tomcat-catalina", "urllib3")])

> 📌 (참고) 기준 목록의 이름은 정규식으로 적습니다(`^curl$`). "curl을 포함하는 이름"으로 찾으면 libcurl과 python3-pycurl까지 걸려 엉뚱한 버전끼리 비교하게 됩니다. 자동 대조에서는 **이름을 정확히 맞추는 일**이 버전 비교만큼 중요합니다(과제 1).

> **더 해 보기: 재부팅을 기다리는 커널**
> 수정된 커널을 설치해도 재부팅하기 전까지는 취약한 커널이 실행 중입니다. 수집 결과의 `os.kernel`은 지금 실행 중인 커널(`uname -r`)입니다. 실제 Ubuntu 서버(VM)라면 설치된 커널과 재부팅 필요 여부를 함께 보고, 둘이 다르면 패치가 "끝나지 않은" 서버로 분류합니다. 컨테이너와 WSL은 호스트의 커널을 쓰므로 이 확인이 의미가 없습니다.
>
> ```bash
> uname -r                                                  # 실행 중인 커널
> dpkg -l 'linux-image-[0-9]*' | awk '/^ii/{print $2, $3}'  # 설치된 커널
> cat /var/run/reboot-required 2>/dev/null || echo "재부팅 필요 없음"
> ```

## 5. 매일 자동으로 돌게 만들기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 구성 | ✅ 완료 |
| 1 | (선택) Windows에서 수집 | ✅ 완료 |
| 2 | 실습용 Ubuntu 서버 만들기 | ✅ 완료 |
| 3 | Ubuntu 서버에서 수집 | ✅ 완료 |
| 4 | 여러 서버의 결과를 합치고 기준 버전과 대조 | ✅ 완료 |
| 5 | 매일 자동으로 돌게 만들기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

수집을 사람이 기억해서 실행하면 자동화가 아닙니다. 서버마다 예약 작업으로 등록하고, 결과를 중앙 저장소로 보냅니다.

```powershell
# Windows Server: 매일 새벽 2시에 실행하고 결과를 공유 폴더로 보냄 (관리자 권한)
schtasks /Create /TN "SW-Inventory" /SC DAILY /ST 02:00 /RU SYSTEM `
  /TR "powershell -ExecutionPolicy Bypass -File C:\sw-inventory\collect_windows.ps1 -OutDir \\inventory-server\inbox"
```

```bash
# Ubuntu: root의 crontab에 등록하고, 결과를 중앙 서버로 전송
sudo crontab -e
0 2 * * * /opt/sw-inventory/collect_linux.sh /var/lib/sw-inventory && scp /var/lib/sw-inventory/*.json inventory@inventory-server:/srv/inbox/
```

중앙 서버에서는 매일 `merge_report.py`를 돌려 통합 목록과 기준 미달 목록을 만들고, 9-4 자동 패치 순환의 ② 단계(자산 대조)와 ③ 단계(패치 작업 생성)로 넘깁니다. 실무에서는 이 역할을 자산·취약점 관리 도구의 에이전트가 맡지만, 원리는 이 실습과 같습니다.

## 6. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 구성 | ✅ 완료 |
| 1 | (선택) Windows에서 수집 | ✅ 완료 |
| 2 | 실습용 Ubuntu 서버 만들기 | ✅ 완료 |
| 3 | Ubuntu 서버에서 수집 | ✅ 완료 |
| 4 | 여러 서버의 결과를 합치고 기준 버전과 대조 | ✅ 완료 |
| 5 | 매일 자동으로 돌게 만들기 | ✅ 완료 |
| 6 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 정리 | 예정 |

### 6.1. 체크포인트

- [ ] 레지스트리·dpkg 목록에는 없지만 포트 대기 프로세스나 SBOM 스캔으로 찾은 SW가 있었나요? 우리 서버에서 WAS와 pip·npm 라이브러리는 어느 폴더에 설치되어 있고, 그 폴더가 수집 대상에 들어 있나요?
- [ ] 호스트명이 겹치는 기기가 있었다면, 우리 회사 자산 목록은 무엇으로 기기를 구분하고 있나요?
- [ ] 기준 버전 목록(`watchlist.csv`)을 사람이 관리한다면 어떤 문제가 생길까요? 어떤 외부 정보로 자동으로 채울 수 있을까요?
- [ ] 수집 스크립트가 출처를 모르는 파일을 root로 실행하면 어떤 일이 생길 수 있을까요?

### 6.2. 과제 1: 기준 목록의 이름 정확히 맞추기

"Python 3.12는 3.12.3 이상이어야 한다"는 기준을 더한다고 합시다. 아래 셀의 `RULE`은 이름을 `python3`으로만 적었습니다. 실행해서 어떤 패키지가 걸리는지 보고, **인터프리터 패키지 하나만** 걸리도록 정규식을 고칩니다.

- 수집 결과의 dpkg 목록에서 Python 인터프리터 패키지의 정확한 이름을 먼저 찾아보세요.

In [ ]:
import re

RULE = {"name_regex": "python3", "min_version": "3.12.3", "reason": "예시 기준"}   # TODO: 정규식을 고칩니다

hits = sorted({(p["name"], p["version"]) for p in inv["packages"] if re.search(RULE["name_regex"], p["name"], re.I)})
print(f"'{RULE['name_regex']}'에 걸린 이름: {len(hits)}개")
table(["이름", "버전"], hits[:15])

<details>
<summary>💡 힌트 보기</summary>

`python3`만 적으면 `python3-minimal`, `libpython3.12-stdlib`, `python3-venv`처럼 이름에 python3가 들어간 모든 패키지와, SBOM으로 찾은 pip 패키지까지 걸릴 수 있습니다. 인터프리터는 `python3.12` 패키지이므로 다음처럼 앞뒤를 고정합니다.

```python
RULE = {"name_regex": "^python3\\.12$", "min_version": "3.12.3", "reason": "예시 기준"}
```

같은 SW라도 배포판마다 패키지 이름이 다르고(`python3.12` / `python312`), 버전 표기도 다릅니다(`3.12.3-1ubuntu0.8`). 그래서 기준 목록은 외부 정보(CVE·KEV)에서 자동으로 만들더라도 **배포판별 이름 매핑**을 함께 관리해야 합니다.
</details>

### 6.3. 과제 2: 기준 목록 자동으로 채우기 (서술)

`watchlist.csv`를 사람이 관리하면 새 취약점이 나올 때마다 늦게 반영됩니다. 아래 질문에 답해 보세요.

1. 8장 실습 1의 CISA KEV 목록과 9장 실습 3의 endoflife.date를 어떻게 쓰면 이 목록을 자동으로 채울 수 있을까요?
2. 자동으로 채운 기준에 걸린 서버가 나오면, 9장 실습 4의 패치 티켓으로 어떻게 넘길까요?

✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

1.
2.

## 7. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 구성 | ✅ 완료 |
| 1 | (선택) Windows에서 수집 | ✅ 완료 |
| 2 | 실습용 Ubuntu 서버 만들기 | ✅ 완료 |
| 3 | Ubuntu 서버에서 수집 | ✅ 완료 |
| 4 | 여러 서버의 결과를 합치고 기준 버전과 대조 | ✅ 완료 |
| 5 | 매일 자동으로 돌게 만들기 | ✅ 완료 |
| 6 | 체크포인트와 과제 | ✅ 완료 |
| 7 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [ ]:
%%bash
docker rm -f web01 web01-clone >/dev/null 2>&1
docker rmi sw-lab-server:clone >/dev/null 2>&1
docker volume rm sw-lab-trivy-cache >/dev/null 2>&1
rm -rf ~/sw-inventory-lab && echo "실습 서버·이미지·볼륨·폴더 삭제됨"

---
> 다음 실습 2에서는 서버의 SW 목록이 아니라 요청 하나의 흐름을 Trace ID로 추적합니다. 9장 실습 3과 4는 이 실습의 "현황 수집"을 클라우드 관리형 서비스로 넓히고, 찾은 대상을 실제로 패치하고 확인합니다.